# SPEC 002 — nnU-Net preprocessing, shard 3 of 4 (Kaggle, CPU only)

The preprocessed KiTS23 data is about 50 GB, but one notebook can save only about
20 GB. So 4 notebooks each preprocess a quarter of the cases, following the shard
plan written by `kits23-prepare-data` (D-017). Each case runs in its own process,
2 at a time; a case that crashes is retried alone. **No GPU quota.**

**Before running**
1. *Add Input*: the outputs of `kits23-stage-1..3` and `kits23-prepare-data`.
2. *Settings*: **Accelerator = None**, **Internet = On**, keep it **Private**.
3. **Save Version → Save & Run All (Commit)**. Expected: about 2 hours.

The training notebook (SPEC 003) attaches all 4 shard outputs.

Generated by `scripts/notebooks/build_preprocess_notebook.py`; edit the repository, not this notebook.

In [ ]:
import os, sys, subprocess
for sub in ("src/project3d/data", "scripts/data"):
    os.makedirs(f"/kaggle/working/project23d/{sub}", exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nnunetv2==2.8.1", "nibabel"],
               check=True)

Write the project code (generated from the repository).

In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/__init__.py
"""Project_23D: renal CT 3D segmentation."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/__init__.py
"""KiTS23 data pipeline (SPEC 002): verify, features, split, nnU-Net export."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/manifest.py
"""Locate KiTS23 files and verify them against the pinned checksum manifest.

The manifest is ``data/sources/kits23_v0.1.4_files.tsv`` (D-011): one row per
file with ``case_id, kind, size_bytes, sha256, url``. ``kind`` is ``image`` or
``segmentation``.
"""

from __future__ import annotations

import csv
import hashlib
import re
from dataclasses import dataclass
from pathlib import Path

KINDS = ("image", "segmentation")
CASE_FILE_RE = re.compile(r"^(case_\d{5})\.nii(\.gz)?$")

# Verification statuses. Only OK is a pass; DECOMPRESSED is a warning (the file
# exists but the platform un-gzipped it, so the published hash cannot apply).
OK = "OK"
MISSING = "MISSING"
SIZE_MISMATCH = "SIZE_MISMATCH"
HASH_MISMATCH = "HASH_MISMATCH"
DECOMPRESSED = "DECOMPRESSED_UNVERIFIED"
DUPLICATE = "DUPLICATE"
FAILING = {MISSING, SIZE_MISMATCH, HASH_MISMATCH, DUPLICATE}


@dataclass(frozen=True)
class Expected:
    """One manifest row: the size and SHA-256 a downloaded file must have."""

    case_id: str
    kind: str
    size_bytes: int
    sha256: str


def load_manifest(tsv_path: str | Path) -> dict[tuple[str, str], Expected]:
    """Read the checksum manifest, keyed by ``(case_id, kind)``."""
    out: dict[tuple[str, str], Expected] = {}
    with open(tsv_path, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh, delimiter="\t"):
            if row["kind"] not in KINDS:
                raise ValueError(f"unknown kind {row['kind']!r} for {row['case_id']}")
            key = (row["case_id"], row["kind"])
            if key in out:
                raise ValueError(f"duplicate manifest row {key}")
            out[key] = Expected(row["case_id"], row["kind"], int(row["size_bytes"]), row["sha256"].lower())
    return out


def find_case_files(root: str | Path) -> dict[str, list[Path]]:
    """Recursively find ``case_XXXXX.nii.gz`` (or ``.nii``) files under ``root``.

    Returns every path per case so duplicates can be reported, not silently picked.
    """
    found: dict[str, list[Path]] = {}
    for path in sorted(Path(root).rglob("case_*")):
        m = CASE_FILE_RE.match(path.name)
        if m and path.is_file():
            found.setdefault(m.group(1), []).append(path)
    return found


def sha256_file(path: str | Path, chunk: int = 8 * 1024 * 1024) -> str:
    """SHA-256 of a file, read in chunks so large scans don't fill memory."""
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        while block := fh.read(chunk):
            digest.update(block)
    return digest.hexdigest()


def _check_file(path: Path, expected: Expected, check_hash: bool) -> dict:
    """Status, size and hash of the one file found for a manifest entry."""
    result = {"path": str(path), "status": OK, "size_bytes": path.stat().st_size, "sha256": ""}
    if path.suffix != ".gz":
        result["status"] = DECOMPRESSED
    elif result["size_bytes"] != expected.size_bytes:
        result["status"] = SIZE_MISMATCH
    elif check_hash:
        result["sha256"] = sha256_file(path)
        if result["sha256"] != expected.sha256:
            result["status"] = HASH_MISMATCH
    return result


def verify_kind(
    manifest: dict[tuple[str, str], Expected],
    kind: str,
    root: str | Path,
    check_hash: bool = True,
) -> list[dict]:
    """Check every manifest entry of one kind against the files under ``root``.

    Returns one report row per manifest entry, plus one row per file found
    under ``root`` that the manifest does not list (status ``UNEXPECTED``).
    """
    found = find_case_files(root)
    rows = []
    for (case_id, entry_kind), expected in sorted(manifest.items()):
        if entry_kind != kind:
            continue
        paths = found.pop(case_id, [])
        row = {"case_id": case_id, "kind": kind, "path": "", "status": OK,
               "size_bytes": "", "sha256": ""}
        if not paths:
            row["status"] = MISSING
        elif len(paths) > 1:
            row.update(status=DUPLICATE, path=";".join(str(p) for p in paths))
        else:
            row.update(_check_file(paths[0], expected, check_hash))
        rows.append(row)
    for case_id, paths in sorted(found.items()):
        for path in paths:
            rows.append({"case_id": case_id, "kind": kind, "path": str(path),
                         "status": "UNEXPECTED", "size_bytes": path.stat().st_size, "sha256": ""})
    return rows


def summarize(rows: list[dict]) -> dict[str, int]:
    """Number of report rows per status."""
    counts: dict[str, int] = {}
    for row in rows:
        counts[row["status"]] = counts.get(row["status"], 0) + 1
    return counts


def resolved_paths(rows: list[dict]) -> dict[str, Path]:
    """Map case_id to its single verified path (OK or DECOMPRESSED rows only)."""
    return {r["case_id"]: Path(r["path"]) for r in rows if r["status"] in (OK, DECOMPRESSED)}


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/features.py
"""Per-case label features used to stratify the split (D-012).

Only segmentations are read. Labels: 0 background, 1 kidney, 2 tumor, 3 cyst.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np

VALID_LABELS = {0, 1, 2, 3}
KITS21_LAST_ID = 299  # cases 0-299 are the KiTS21 cohort; 400+ were added in KiTS23


def cohort_of(case_id: str) -> str:
    """Cohort proxy for contrast phase (D-010, D-012); not a phase label."""
    return "kits21" if int(case_id.split("_")[1]) <= KITS21_LAST_ID else "kits23_new"


def label_features(seg: np.ndarray, spacing_mm: tuple[float, ...]) -> dict:
    """Volumes in millilitres and presence flags from a label array."""
    present = {int(v) for v in np.unique(seg)}
    unknown = present - VALID_LABELS
    if unknown:
        raise ValueError(f"unexpected label values {sorted(unknown)}")
    counts = np.bincount(seg.ravel().astype(np.int64), minlength=4)
    voxel_ml = float(np.prod(spacing_mm)) / 1000.0
    return {
        "kidney_ml": round(counts[1] * voxel_ml, 3),
        "tumor_ml": round(counts[2] * voxel_ml, 3),
        "cyst_ml": round(counts[3] * voxel_ml, 3),
        "has_tumor": bool(counts[2] > 0),
        "has_cyst": bool(counts[3] > 0),
    }


def case_features(case_id: str, seg_path: str | Path) -> dict:
    """Read one segmentation and return its features plus header geometry."""
    import nibabel as nib

    img = nib.load(str(seg_path))
    seg = np.asanyarray(img.dataobj)
    if not np.all(np.mod(seg, 1) == 0):
        raise ValueError(f"{case_id}: non-integer label values")
    seg = seg.astype(np.uint8)
    spacing = tuple(float(z) for z in img.header.get_zooms()[:3])
    row = {"case_id": case_id, "cohort": cohort_of(case_id),
           "shape": "x".join(map(str, seg.shape)),
           "spacing_mm": "x".join(f"{s:.4f}" for s in spacing)}
    row.update(label_features(seg, spacing))
    return row


def geometry_mismatch(image_path: str | Path, seg_path: str | Path, atol: float = 1e-3) -> str | None:
    """Compare image and mask headers (no voxel data read). None if they match."""
    import nibabel as nib

    img, seg = nib.load(str(image_path)), nib.load(str(seg_path))
    if img.shape[:3] != seg.shape[:3]:
        return f"shape {img.shape[:3]} vs {seg.shape[:3]}"
    if not np.allclose(img.affine, seg.affine, atol=atol):
        return "affine differs"
    return None


def add_size_bins(rows: list[dict]) -> tuple[float, float]:
    """Add ``size_bin`` in place: tertiles of tumor volume among tumor cases.

    Cases without tumor get ``none``. Returns the two cut points (ml) so they
    can be recorded with the split.
    """
    vols = np.array([r["tumor_ml"] for r in rows if r["has_tumor"]], dtype=float)
    if vols.size == 0:
        cuts = (0.0, 0.0)
    else:
        q1, q2 = np.quantile(vols, [1 / 3, 2 / 3])
        cuts = (float(q1), float(q2))
    for r in rows:
        if not r["has_tumor"]:
            r["size_bin"] = "none"
        elif r["tumor_ml"] <= cuts[0]:
            r["size_bin"] = "small"
        elif r["tumor_ml"] <= cuts[1]:
            r["size_bin"] = "medium"
        else:
            r["size_bin"] = "large"
    return cuts


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/split.py
"""Case-level stratified train/validation/test split (SPEC 002, D-012).

Strata are ``cohort | size_bin | cyst``. Per-stratum counts are allocated by
largest-remainder apportionment, so the global counts hit the target fractions
exactly and every stratum is represented proportionally. Cases are shuffled
within each stratum with a fixed seed.

The saved ``split.json`` is the authority. Its fingerprint is recorded so any
later change is detectable; the split is never regenerated after training.
"""

from __future__ import annotations

import hashlib
import json
import math
import random

ROLES = ("train", "val", "test")


def stratum_key(row: dict) -> str:
    """The stratum a case belongs to, e.g. ``kits23|small|nocyst``."""
    return f"{row['cohort']}|{row['size_bin']}|{'cyst' if row['has_cyst'] else 'nocyst'}"


def _apportion(sizes: dict[str, int], total: int, capacity: dict[str, int]) -> dict[str, int]:
    """Split ``total`` across strata proportionally to ``sizes``, within ``capacity``."""
    n = sum(sizes.values())
    if n == 0 or total == 0:
        return {k: 0 for k in sizes}
    quota = {k: sizes[k] * total / n for k in sizes}
    alloc = {k: min(math.floor(quota[k]), capacity[k]) for k in sizes}
    remaining = total - sum(alloc.values())
    # Largest remainder first; ties broken by stratum name for determinism.
    order = sorted(sizes, key=lambda k: (-(quota[k] - math.floor(quota[k])), k))
    while remaining > 0:
        progressed = False
        for k in order:
            if remaining == 0:
                break
            if alloc[k] < capacity[k]:
                alloc[k] += 1
                remaining -= 1
                progressed = True
        if not progressed:
            raise ValueError("not enough cases to satisfy the requested split")
    return alloc


def make_split(rows: list[dict], seed: int = 42,
               fractions: tuple[float, float, float] = (0.70, 0.15, 0.15)) -> dict[str, str]:
    """Return ``{case_id: role}`` with role in ``train``/``val``/``test``."""
    if abs(sum(fractions) - 1.0) > 1e-9:
        raise ValueError("fractions must sum to 1")
    ids = [r["case_id"] for r in rows]
    if len(set(ids)) != len(ids):
        raise ValueError("duplicate case_id in feature rows")

    strata: dict[str, list[str]] = {}
    for r in rows:
        strata.setdefault(stratum_key(r), []).append(r["case_id"])
    sizes = {k: len(v) for k, v in strata.items()}
    n = len(rows)
    n_test = round(n * fractions[2])
    n_val = round(n * fractions[1])

    test_alloc = _apportion(sizes, n_test, sizes)
    val_alloc = _apportion(sizes, n_val, {k: sizes[k] - test_alloc[k] for k in sizes})

    rng = random.Random(seed)
    roles: dict[str, str] = {}
    for key in sorted(strata):
        members = sorted(strata[key])
        rng.shuffle(members)
        t, v = test_alloc[key], val_alloc[key]
        for i, cid in enumerate(members):
            roles[cid] = "test" if i < t else "val" if i < t + v else "train"
    return dict(sorted(roles.items()))


def fingerprint(roles: dict[str, str]) -> str:
    """SHA-256 of the canonical ``{case_id: role}`` mapping."""
    canon = json.dumps(dict(sorted(roles.items())), sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


def nnunet_splits(roles: dict[str, str]) -> list[dict[str, list[str]]]:
    """nnU-Net ``splits_final.json`` content: one fold, train vs val. Test is excluded."""
    return [{
        "train": sorted(c for c, r in roles.items() if r == "train"),
        "val": sorted(c for c, r in roles.items() if r == "val"),
    }]


def summary(rows: list[dict], roles: dict[str, str]) -> dict:
    """Counts per role, overall and by each stratification variable."""
    out: dict = {"total": {r: 0 for r in ROLES}}
    for var in ("cohort", "size_bin", "has_cyst"):
        out[var] = {}
    for row in rows:
        role = roles[row["case_id"]]
        out["total"][role] += 1
        for var in ("cohort", "size_bin", "has_cyst"):
            bucket = out[var].setdefault(str(row[var]), {r: 0 for r in ROLES})
            bucket[role] += 1
    return out


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/nnunet_export.py
"""Build the nnU-Net v2 raw dataset folder for KiTS23 (SPEC 002).

Mirrors nnU-Net's own ``dataset_conversion/Dataset220_KiTS2023.py``: region-based
labels (kidney+masses, masses, tumor), ``regions_class_order`` (1, 3, 2) and the
``NibabelIOWithReorient`` reader. Two differences, both deliberate:

- only train and validation cases go into ``imagesTr``/``labelsTr``, so the
  locked test cases cannot influence nnU-Net's planning or normalization;
- files are symlinked by default instead of copied, so no extra disk is used.
"""

from __future__ import annotations

import json
import os
import shutil
from pathlib import Path

DATASET_ID = 220
DATASET_NAME = "KiTS2023"
DATASET_JSON = {
    "channel_names": {"0": "CT"},
    "labels": {"background": 0, "kidney": [1, 2, 3], "masses": [2, 3], "tumor": 2},
    "regions_class_order": [1, 3, 2],
    "overwrite_image_reader_writer": "NibabelIOWithReorient",
    "name": DATASET_NAME,
    "release": "0.1.4",
    "reference": "https://github.com/neheller/kits23",
    "licence": "CC BY-NC-SA 4.0 (KiTS23 data terms)",
    "description": "KiTS23 train+val cases only; locked test cases excluded (D-012)",
}


def dataset_folder_name(dataset_id: int = DATASET_ID, name: str = DATASET_NAME) -> str:
    """nnU-Net's folder name for a dataset, e.g. ``Dataset220_KiTS2023``."""
    return f"Dataset{dataset_id:03d}_{name}"


def _file_ending(paths: list[Path]) -> str:
    endings = {".nii.gz" if p.name.endswith(".nii.gz") else p.suffix for p in paths}
    if len(endings) != 1:
        raise ValueError(f"mixed file endings {sorted(endings)}; nnU-Net needs one")
    return endings.pop()


def _place(src: Path, dst: Path, mode: str) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    if mode == "symlink":
        os.symlink(src.resolve(), dst)
    elif mode == "copy":
        shutil.copy2(src, dst)
    else:
        raise ValueError(f"unknown mode {mode!r}")


def build_raw_dataset(roles: dict[str, str], images: dict[str, Path], segs: dict[str, Path],
                      raw_root: str | Path, mode: str = "symlink",
                      dataset_id: int = DATASET_ID) -> Path:
    """Create ``<raw_root>/DatasetXXX_KiTS2023`` and return its path.

    ``imagesTr``/``labelsTr`` hold train+val cases; ``imagesTs`` holds test images.
    Test labels are never placed in the nnU-Net folder.
    """
    trainval = sorted(c for c, r in roles.items() if r in ("train", "val"))
    test = sorted(c for c, r in roles.items() if r == "test")
    missing = [c for c in roles if c not in images or (c in trainval and c not in segs)]
    if missing:
        raise ValueError(f"no verified file for {len(missing)} cases, e.g. {missing[:3]}")

    ending = _file_ending([images[c] for c in roles] + [segs[c] for c in trainval])
    out = Path(raw_root) / dataset_folder_name(dataset_id)
    if out.exists():
        shutil.rmtree(out)
    for c in trainval:
        _place(images[c], out / "imagesTr" / f"{c}_0000{ending}", mode)
        _place(segs[c], out / "labelsTr" / f"{c}{ending}", mode)
    for c in test:
        _place(images[c], out / "imagesTs" / f"{c}_0000{ending}", mode)

    meta = dict(DATASET_JSON, numTraining=len(trainval), file_ending=ending)
    (out / "dataset.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return out


def check_no_test_leak(dataset_dir: str | Path, roles: dict[str, str]) -> None:
    """Fail if any test case appears in imagesTr or labelsTr."""
    test = {c for c, r in roles.items() if r == "test"}
    for sub in ("imagesTr", "labelsTr"):
        for p in (Path(dataset_dir) / sub).iterdir():
            if p.name[:10] in test:
                raise AssertionError(f"test case {p.name[:10]} found in {sub}")


In [ ]:
%%writefile /kaggle/working/project23d/scripts/data/prepare_kits23.py
"""SPEC 002 pipeline: verify -> features -> split -> nnU-Net raw dataset.

Runs on a Kaggle CPU session (or locally on a small subset). Example::

    python scripts/data/prepare_kits23.py --input-root /kaggle/input \
        --manifest data/sources/kits23_v0.1.4_files.tsv \
        --out-dir /kaggle/working/spec002 --raw-root /tmp/nnUNet_raw

Outputs in ``--out-dir``: verify_report.csv, case_features.csv, split.json,
splits_final.json. The nnU-Net raw dataset is written under ``--raw-root``.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import shutil
import statistics
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

from project3d.data import features, manifest, nnunet_export, split

IMAGE_MIN_BYTES = 10 * 1024 * 1024  # CT volumes are tens of MB; label files are ~1 MB


def detect_roots(input_root: Path, min_files: int = 1,
                 link_dir: Path | None = None, link_mode: str = "symlink") -> tuple[Path, Path]:
    """Find the images folder and the segmentations folder by typical file size.

    Segmentation folders with fewer than ``min_files`` case files are ignored (e.g.
    nnU-Net's validation predictions sitting next to the full dataset). Images may
    be split over several folders (one per Kaggle staging shard, D-015); they are
    then linked into ``link_dir/images`` so the pipeline sees one folder.
    """
    by_dir: dict[Path, list[int]] = {}
    for paths in manifest.find_case_files(input_root).values():
        for p in paths:
            by_dir.setdefault(p.parent, []).append(p.stat().st_size)
    if not by_dir:
        raise SystemExit(f"no case_XXXXX.nii(.gz) files found under {input_root}")
    big = sorted(d for d, s in by_dir.items() if statistics.median(s) >= IMAGE_MIN_BYTES)
    small = [d for d, s in by_dir.items()
             if statistics.median(s) < IMAGE_MIN_BYTES and len(s) >= min_files]
    n_images = sum(len(by_dir[d]) for d in big)
    if not big or n_images < min_files or len(small) != 1:
        raise SystemExit(f"could not tell images from segmentations; folders found: "
                         f"{ {str(d): len(s) for d, s in by_dir.items()} }. "
                         f"Pass --images-root and --segs-root explicitly.")
    if len(big) == 1:
        return big[0], small[0]
    if link_dir is None:
        raise SystemExit(f"images are split over {len(big)} folders; pass a link folder")
    merged = Path(link_dir) / "images"
    if merged.exists():
        shutil.rmtree(merged)
    for d in big:
        for p in sorted(d.iterdir()):
            if not manifest.CASE_FILE_RE.match(p.name):
                continue
            if (merged / p.name).exists():
                raise SystemExit(f"{p.name} appears in more than one image folder")
            nnunet_export._place(p, merged / p.name, link_mode)
    print(f"linked {n_images} images from {len(big)} folders into {merged}")
    return merged, small[0]


def write_csv(path: Path, rows: list[dict]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--manifest", type=Path, required=True)
    ap.add_argument("--input-root", type=Path, help="auto-detect images/segmentation folders here")
    ap.add_argument("--images-root", type=Path)
    ap.add_argument("--segs-root", type=Path)
    ap.add_argument("--out-dir", type=Path, required=True)
    ap.add_argument("--raw-root", type=Path, required=True)
    ap.add_argument("--link-mode", choices=("symlink", "copy"), default="symlink")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--subset", action="store_true",
                    help="allow missing cases (local smoke test); never use for the real run")
    ap.add_argument("--skip-hash", action="store_true", help="size check only (faster, weaker)")
    args = ap.parse_args(argv)

    t0 = time.time()
    args.out_dir.mkdir(parents=True, exist_ok=True)
    if args.images_root and args.segs_root:
        images_root, segs_root = args.images_root, args.segs_root
    elif args.input_root:
        images_root, segs_root = detect_roots(args.input_root, link_dir=args.raw_root.parent / "kits23_merged",
                                              link_mode=args.link_mode)
    else:
        ap.error("give --input-root, or both --images-root and --segs-root")
    print(f"images: {images_root}\nsegmentations: {segs_root}")

    # 1. Verify every file against the pinned manifest.
    exp = manifest.load_manifest(args.manifest)
    report = (manifest.verify_kind(exp, "image", images_root, not args.skip_hash)
              + manifest.verify_kind(exp, "segmentation", segs_root, not args.skip_hash))
    write_csv(args.out_dir / "verify_report.csv", report)
    counts = manifest.summarize(report)
    print(f"[1/4] verify: {counts}  ({time.time() - t0:.0f}s)")
    failing = {s: n for s, n in counts.items() if s in manifest.FAILING or s == "UNEXPECTED"}
    if args.subset:
        failing.pop(manifest.MISSING, None)
    if failing:
        print("STOP: verification failed; see verify_report.csv")
        return 1
    if counts.get(manifest.DECOMPRESSED):
        print("WARNING: some files were un-gzipped by the platform; their hashes cannot be checked")

    images = manifest.resolved_paths([r for r in report if r["kind"] == "image"])
    segs = manifest.resolved_paths([r for r in report if r["kind"] == "segmentation"])
    cases = sorted(set(images) & set(segs))
    if not args.subset and len(cases) != len({c for c, _ in exp}):
        print(f"STOP: only {len(cases)} cases have both image and label")
        return 1

    # 2. Image/mask geometry must agree (SPEC 001 5.4), then label features per case.
    bad = {c: why for c in cases if (why := features.geometry_mismatch(images[c], segs[c]))}
    if bad:
        print(f"STOP: image/mask geometry mismatch: {bad}")
        return 1
    rows = [features.case_features(c, segs[c]) for c in cases]
    cuts = features.add_size_bins(rows)
    write_csv(args.out_dir / "case_features.csv", rows)
    print(f"[2/4] features: {len(rows)} cases, tumor-size tertile cuts (ml) = "
          f"{cuts[0]:.2f}, {cuts[1]:.2f}  ({time.time() - t0:.0f}s)")

    # 3. Split once, fingerprint, save.
    roles = split.make_split(rows, seed=args.seed)
    record = {
        "spec": "SPEC 002",
        "dataset": "KiTS23 v0.1.4",
        # Line endings normalized so Windows and Kaggle copies hash the same.
        "manifest_sha256": hashlib.sha256(
            args.manifest.read_bytes().replace(b"\r\n", b"\n")).hexdigest(),
        "seed": args.seed,
        "fractions": {"train": 0.70, "val": 0.15, "test": 0.15},
        "strata": "cohort | size_bin | cyst",
        "size_bin_cuts_ml": list(cuts),
        "fingerprint_sha256": split.fingerprint(roles),
        "summary": split.summary(rows, roles),
        "roles": roles,
    }
    (args.out_dir / "split.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    (args.out_dir / "splits_final.json").write_text(
        json.dumps(split.nnunet_splits(roles), indent=2), encoding="utf-8")
    print(f"[3/4] split: {record['summary']['total']}  fingerprint {record['fingerprint_sha256'][:16]}...")

    # 4. nnU-Net raw dataset (train+val only in imagesTr).
    ds = nnunet_export.build_raw_dataset(roles, images, segs, args.raw_root, mode=args.link_mode)
    nnunet_export.check_no_test_leak(ds, roles)
    print(f"[4/4] nnU-Net raw dataset: {ds}  ({time.time() - t0:.0f}s total)")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/project23d/scripts/data/preprocess_shard.py
"""SPEC 002, part 2: run nnU-Net preprocessing for one shard of cases (D-017).

Needs the SPEC 002 prepare output (``nnUNetPlans.json``, ``dataset.json`` and
``shard_plan.json``) and the KiTS23 staging outputs under ``--input-root``. Each
case runs in its own process, ``--workers`` at a time, so memory is returned after
every case. A case whose process dies is retried once with nothing else running.
Cases already written are skipped, so a rerun picks up where it stopped. Example::

    python scripts/data/preprocess_shard.py --input-root /kaggle/input --shard 1 \
        --out-dir /kaggle/working/nnUNet_preprocessed/Dataset220_KiTS2023/nnUNetPlans_3d_fullres \
        --record /kaggle/working/spec002/shard_1.json
"""

from __future__ import annotations

import argparse
import json
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from prepare_kits23 import detect_roots, manifest  # also puts src/ on sys.path

DATASET = "Dataset220_KiTS2023"
CONFIG = "3d_fullres"
N_CASES = 489
SUFFIXES = (".b2nd", "_seg.b2nd", ".pkl")


def find_plans_dir(root: Path) -> Path:
    hits = sorted(p.parent for p in root.rglob("shard_plan.json")
                  if p.parent.name == DATASET and (p.parent / "nnUNetPlans.json").exists())
    if len(hits) != 1:
        raise SystemExit(f"need exactly one {DATASET} folder with nnUNetPlans.json and shard_plan.json, found {hits}")
    return hits[0]


def is_done(out_dir: Path, case: str) -> bool:
    return all((out_dir / f"{case}{s}").exists() for s in SUFFIXES)


def preprocess_one(case: str, image: str, seg: str, plans_dir: Path, out_dir: Path) -> None:
    from batchgenerators.utilities.file_and_folder_operations import load_json
    from nnunetv2.utilities.plans_handling.plans_handler import PlansManager

    pm = PlansManager(load_json(str(plans_dir / "nnUNetPlans.json")))
    cm = pm.get_configuration(CONFIG)
    pp = cm.preprocessor_class(verbose=False)
    pp.run_case_save(str(out_dir / case), [image], seg, pm, cm, load_json(str(plans_dir / "dataset.json")))


def run_case(case: str, image: Path, seg: Path, plans_dir: Path, out_dir: Path) -> tuple[str, int, float]:
    t = time.time()
    code = subprocess.run([sys.executable, __file__, "--one", case, str(image), str(seg),
                           str(plans_dir), str(out_dir)]).returncode
    if code == 0 and not is_done(out_dir, case):
        code = -1
    print(f"{case}: {'ok' if code == 0 else f'FAILED ({code})'} in {time.time() - t:.0f} s", flush=True)
    return case, code, time.time() - t


def main(argv: list[str] | None = None) -> int:
    if argv is None and len(sys.argv) > 1 and sys.argv[1] == "--one":
        case, image, seg, plans_dir, out_dir = sys.argv[2:7]
        preprocess_one(case, image, seg, Path(plans_dir), Path(out_dir))
        return 0
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--input-root", type=Path, required=True)
    ap.add_argument("--shard", type=int, required=True, help="1-based")
    ap.add_argument("--out-dir", type=Path, required=True)
    ap.add_argument("--record", type=Path, required=True)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--link-dir", type=Path, default=Path("/tmp/kits23_merged"))
    args = ap.parse_args(argv)
    t0 = time.time()

    plans_dir = find_plans_dir(args.input_root)
    plan = json.loads((plans_dir / "shard_plan.json").read_text())
    cases = plan["shards"][args.shard - 1]
    images_root, segs_root = detect_roots(args.input_root, min_files=N_CASES, link_dir=args.link_dir)
    images, segs = manifest.find_case_files(images_root), manifest.find_case_files(segs_root)
    args.out_dir.mkdir(parents=True, exist_ok=True)
    todo = [c for c in cases if not is_done(args.out_dir, c)]
    print(f"shard {args.shard}/{plan['n_shards']}: {len(cases)} cases, {len(todo)} to do, "
          f"{args.workers} workers", flush=True)

    def job(c):
        return run_case(c, images[c][0], segs[c][0], plans_dir, args.out_dir)

    with ThreadPoolExecutor(args.workers) as ex:
        results = list(ex.map(job, todo))
    retry = [c for c, code, _ in results if code != 0]
    if retry:
        print(f"retrying {len(retry)} failed case(s) one at a time: {retry}", flush=True)
        results += [job(c) for c in retry]
    failed = [c for c in cases if not is_done(args.out_dir, c)]
    secs = [s for _, code, s in results if code == 0]
    size_gb = sum(f.stat().st_size for f in args.out_dir.iterdir()) / 1e9
    record = {
        "shard": args.shard, "n_shards": plan["n_shards"], "cases": len(cases),
        "done": len(cases) - len(failed), "failed": failed, "workers": args.workers,
        "mean_case_seconds": round(sum(secs) / len(secs), 1) if secs else None,
        "size_gb": round(size_gb, 2), "hours": round((time.time() - t0) / 3600, 2),
    }
    args.record.parent.mkdir(parents=True, exist_ok=True)
    args.record.write_text(json.dumps(record, indent=2))
    print(json.dumps(record, indent=1))
    return 1 if failed else 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
SHARD = 3
W = "/kaggle/working/project23d"
os.environ["PYTHONPATH"] = f"{W}/src"
os.environ.update({k: "/tmp/" + k for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results")})
code = subprocess.run([sys.executable, f"{W}/scripts/data/preprocess_shard.py",
                       "--input-root", "/kaggle/input", "--shard", str(SHARD), "--workers", "2",
                       "--out-dir", "/kaggle/working/nnUNet_preprocessed/Dataset220_KiTS2023/nnUNetPlans_3d_fullres",
                       "--record", f"/kaggle/working/spec002/shard_{SHARD}.json"]).returncode
print("DONE" if code == 0 else "SOME CASES FAILED: see the record above.",
      "Copy the printed record back to Claude.")